# College Town Rental Yield Dashboard
**MSDS 692 — Data Science Practicum I | Ilse Severance**

Interactive dashboard powered by the **Random Forest** model (selected as the best-performing model on the 2024 test set).  
Explore predicted gross rental yields for US college-town counties — **2025 & 2026**.

---
**How to use:** Run all cells, then use the dropdowns and sliders to filter by state, year, confidence level, and number of counties shown. All charts update automatically.

In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import plotly.express as px
import plotly.graph_objects as go

# Load pre-computed predictions — no model re-training needed
df = pd.read_csv('predictions_2025_2026.csv')

# Convert to percentages for display
df['predicted_yield_pct'] = df['predicted_yield'] * 100
df['yield_2024_pct']      = df['yield_2024'] * 100
df['yield_change_pct']    = df['yield_change'] * 100

print(f"Loaded {len(df):,} rows | {df['state'].nunique()} states | {df['county'].nunique()} counties")
df.head(3)

Loaded 1,024 rows | 49 states | 428 counties


,state,county,year,predicted_yield,acs_imputed,confidence,yield_2024,yield_change,predicted_yield_pct,yield_2024_pct,yield_change_pct
0,AL,Baldwin,2025,0.065536,0,high,0.059519,0.006017,6.5536,5.951884,0.601716
1,AL,Baldwin,2026,0.064993,0,high,0.059519,0.005474,6.4993,5.951884,0.547416
2,AL,Jefferson,2025,0.069807,0,high,0.081621,-0.011814,6.9807,8.162108,-1.181408


In [2]:
import sys, os
from IPython.display import display, HTML, clear_output
from plotly.subplots import make_subplots

# ── Load data ─────────────────────────────────────────────────────────────────
# predictions_2025_2026.csv holds Phase 4's BEST_MODEL output, which resolved
# to Random Forest on the latest run (lower test MAE than the tree ensemble).
df_ens = pd.read_csv('predictions_2025_2026.csv')
for col in ['predicted_yield', 'yield_2024', 'yield_change']:
    df_ens[col + '_pct'] = df_ens[col] * 100

MODEL_LBL = 'Random Forest'

# ── Control widgets ───────────────────────────────────────────────────────────
state_dd = widgets.Dropdown(
    options=['All States'] + sorted(df_ens['state'].unique().tolist()),
    value='All States', description='State:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='220px')
)
county_dd = widgets.Dropdown(
    options=['All Counties'], value='All Counties', description='County:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='280px')
)
year_rb = widgets.RadioButtons(
    options=[2025, 2026], value=2025, description='Year:',
    style={'description_width': 'initial'}
)
top_n_sl = widgets.IntSlider(
    min=10, max=50, value=20, step=5, description='Top N counties:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='340px')
)
conf_dd = widgets.Dropdown(
    options=['All'] + sorted(df_ens['confidence'].dropna().unique().tolist()),
    value='All', description='Confidence:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='220px')
)

# ── Feature importance (static) ───────────────────────────────────────────────
CAT_COLORS = {
    'Demographic (ACS)': '#2ca02c', 'College (IPEDS)': '#ff7f0e',
    'Market (Zillow)':   '#1f77b4', 'Macro (FRED)':    '#d62728',
    'Engineered':        '#9467bd', 'Other':           '#7f7f7f',
}
fi_data = pd.DataFrame({
    'feature':    ['median_home_value', 'median_household_income', 'median_gross_rent',
                   'zori_is_real', 'vacancy_rate'],
    'importance': [0.3203, 0.0685, 0.0623, 0.0574, 0.0485],
    'category':   ['Demographic (ACS)', 'Demographic (ACS)', 'Demographic (ACS)',
                   'Other', 'Demographic (ACS)']
})

# ── Output container for charts ───────────────────────────────────────────────
main_out = widgets.Output()

def update(change=None):
    filtered = df_ens[df_ens['year'] == year_rb.value].copy()
    if state_dd.value  != 'All States':   filtered = filtered[filtered['state']  == state_dd.value]
    if county_dd.value != 'All Counties': filtered = filtered[filtered['county'] == county_dd.value]
    if conf_dd.value   != 'All':          filtered = filtered[filtered['confidence'] == conf_dd.value]
    if len(filtered) == 0:
        return

    top_n = top_n_sl.value
    year  = year_rb.value
    top   = filtered.nlargest(top_n, 'predicted_yield').copy()
    top['label']      = top['county'] + ', ' + top['state']
    filtered['label'] = filtered['county'] + ', ' + filtered['state']

    avg_yield     = filtered['predicted_yield_pct'].mean()
    avg_change    = filtered['yield_change_pct'].mean()
    n_counties    = len(filtered)
    n_states      = filtered['state'].nunique()
    pct_improving = (filtered['yield_change_pct'] > 0).mean() * 100
    chg_color     = '#28a745' if avg_change >= 0 else '#dc3545'

    with main_out:
        clear_output(wait=True)

        # KPI row
        display(HTML(
            '<div style="display:flex;gap:40px;padding:16px;background:#f0f4f8;'
            'border-radius:10px;margin:8px 0;font-family:sans-serif">'
            f'<div style="text-align:center"><div style="font-size:26px;font-weight:700;color:#2c7be5">{avg_yield:.2f}%</div>'
            '<div style="color:#555;font-size:12px;margin-top:4px">Avg Predicted Yield</div></div>'
            f'<div style="text-align:center"><div style="font-size:26px;font-weight:700;color:{chg_color}">{avg_change:+.2f}%</div>'
            '<div style="color:#555;font-size:12px;margin-top:4px">Avg Change vs 2024</div></div>'
            f'<div style="text-align:center"><div style="font-size:26px;font-weight:700;color:#17a2b8">{n_states}</div>'
            '<div style="color:#555;font-size:12px;margin-top:4px">States</div></div>'
            f'<div style="text-align:center"><div style="font-size:26px;font-weight:700;color:#6f42c1">{n_counties:,}</div>'
            '<div style="color:#555;font-size:12px;margin-top:4px">Counties</div></div>'
            f'<div style="text-align:center"><div style="font-size:26px;font-weight:700;color:#fd7e14">{pct_improving:.0f}%</div>'
            '<div style="color:#555;font-size:12px;margin-top:4px">Counties Improving</div></div>'
            '</div>'
        ))

        # Feature importance (bar + donut)
        fi_sorted  = fi_data.sort_values('importance')
        cat_totals = fi_data.groupby('category')['importance'].sum().reset_index()
        fi_fig = make_subplots(
            rows=1, cols=2, column_widths=[0.58, 0.42],
            specs=[[{'type': 'xy'}, {'type': 'domain'}]],
            subplot_titles=['Top 5 Feature Importances', 'Importance by Data Source']
        )
        fi_fig.add_trace(go.Bar(
            x=fi_sorted['importance'], y=fi_sorted['feature'], orientation='h',
            marker_color=fi_sorted['category'].map(CAT_COLORS).fillna('gray').tolist(),
            text=[f'{v:.4f}' for v in fi_sorted['importance']],
            textposition='outside', showlegend=False,
            hovertemplate='<b>%{y}</b><br>Importance: %{x:.4f}<extra></extra>'
        ), row=1, col=1)
        fi_fig.add_trace(go.Pie(
            labels=cat_totals['category'], values=cat_totals['importance'],
            hole=0.45,
            marker_colors=cat_totals['category'].map(CAT_COLORS).fillna('gray').tolist(),
            textinfo='percent', textfont_size=12, pull=[0.05] * len(cat_totals),
            hovertemplate='<b>%{label}</b><br>Share: %{percent}<extra></extra>'
        ), row=1, col=2)
        fi_fig.update_layout(height=310, margin=dict(l=10, r=10, t=40, b=10),
                             legend=dict(orientation='h', yanchor='top', y=-0.1,
                                         xanchor='center', x=0.75, font=dict(size=11)))
        display(fi_fig)

        # Chart 1: Top N counties
        t1 = top.sort_values('predicted_yield')
        fig1 = px.bar(t1, x='predicted_yield_pct', y='label', orientation='h',
                      title=f'Top {top_n} Counties — Predicted Rental Yield ({year}) [{MODEL_LBL}]',
                      labels={'predicted_yield_pct': 'Predicted Yield (%)', 'label': ''},
                      color='predicted_yield_pct', color_continuous_scale='Blues',
                      text=t1['predicted_yield_pct'].map(lambda v: f'{v:.2f}%'))
        fig1.update_traces(textposition='outside')
        fig1.update_layout(height=max(400, top_n * 22), coloraxis_showscale=False,
                           margin=dict(l=10, r=70, t=45, b=30))
        display(fig1)

        # Chart 2: Yield change
        t2 = top.sort_values('yield_change')
        fig2 = go.Figure(go.Bar(
            x=t2['yield_change_pct'], y=t2['label'], orientation='h',
            marker_color=['#dc3545' if v < 0 else '#28a745' for v in t2['yield_change_pct']],
            text=[f'{v:+.2f}%' for v in t2['yield_change_pct']], textposition='outside'
        ))
        fig2.update_layout(title=f'Yield Change vs 2024 — Top {top_n} Counties ({year}) [{MODEL_LBL}]',
                           xaxis_title='Change in Yield (%)', yaxis_title='',
                           height=max(400, top_n * 22), margin=dict(l=10, r=70, t=45, b=30))
        fig2.add_vline(x=0, line_dash='dash', line_color='gray', line_width=1)
        display(fig2)

        # Chart 3: State avg
        state_avg = (filtered.groupby('state')
                     .agg(avg_yield=('predicted_yield_pct', 'mean'),
                          avg_change=('yield_change_pct', 'mean'),
                          n_counties=('county', 'count'))
                     .reset_index().sort_values('avg_yield', ascending=False))
        fig3 = px.bar(state_avg, x='state', y='avg_yield',
                      color='avg_yield', color_continuous_scale='Blues',
                      title=f'Average Predicted Yield by State ({year}) [{MODEL_LBL}]',
                      labels={'avg_yield': 'Avg Yield (%)', 'state': 'State', 'avg_change': 'Avg Δ (%)'},
                      text=state_avg['avg_yield'].map(lambda v: f'{v:.2f}%'),
                      hover_data={'n_counties': True, 'avg_change': ':.2f'})
        fig3.update_traces(textposition='outside')
        fig3.update_layout(height=450, margin=dict(l=10, r=10, t=45, b=40))
        display(fig3)

        # Chart 4: Scatter
        fig4 = px.scatter(filtered, x='yield_2024_pct', y='predicted_yield_pct',
                          color='predicted_yield_pct', color_continuous_scale='Blues',
                          hover_data={'state': True, 'county': True, 'confidence': True},
                          title=f'2024 Actual vs {year} Predicted Yield [{MODEL_LBL}]',
                          labels={'yield_2024_pct': '2024 Actual Yield (%)',
                                  'predicted_yield_pct': f'{year} Predicted Yield (%)',
                                  'yield_change_pct': 'Change (%)'},
                          opacity=0.65)
        mn = min(filtered['yield_2024_pct'].min(), filtered['predicted_yield_pct'].min())
        mx = max(filtered['yield_2024_pct'].max(), filtered['predicted_yield_pct'].max())
        fig4.add_shape(type='line', x0=mn, x1=mx, y0=mn, y1=mx,
                       line=dict(dash='dash', color='gray', width=1))
        fig4.update_layout(height=460, margin=dict(l=10, r=10, t=45, b=40))
        display(fig4)

# ── County cascade ────────────────────────────────────────────────────────────
def _refresh_county_opts():
    s = state_dd.value
    county_dd.unobserve(update, names='value')
    county_dd.options = (['All Counties'] if s == 'All States' else
                         ['All Counties'] + sorted(df_ens[df_ens['state'] == s]['county'].unique().tolist()))
    county_dd.value = 'All Counties'
    county_dd.observe(update, names='value')

def on_state_change(change):
    _refresh_county_opts()
    update()

state_dd.observe(on_state_change, names='value')

for w in [county_dd, year_rb, top_n_sl, conf_dd]:
    w.observe(update, names='value')

# ── Layout ────────────────────────────────────────────────────────────────────
controls = widgets.HBox([
    widgets.VBox([state_dd, county_dd, conf_dd],
                 layout=widgets.Layout(padding='8px', margin='0 20px 0 0')),
    widgets.VBox([year_rb, top_n_sl], layout=widgets.Layout(padding='8px'))
], layout=widgets.Layout(border='1px solid #dee2e6', border_radius='8px',
                          padding='8px', margin='8px 0'))

title_html = widgets.HTML(
    '<h2 style="font-family:sans-serif;color:#2c3e50;margin:0">'
    'College Town Rental Yield Dashboard</h2>'
    '<p style="font-family:sans-serif;color:#666;margin:4px 0 8px">'
    'Predicted gross rental yields (Random Forest) | 2025–2026</p>'
)

vbox = widgets.VBox([title_html, controls, main_out])

# ── Close any previous instance so it vanishes from wherever it was rendered,
#    then display the fresh one. This handles both cases: VS Code clearing the
#    cell output (previous widget already gone, close() is a no-op) and VS Code
#    keeping it (close() removes it from the widget registry so it disappears).
_KEY = '_ry_dashboard'
prev = sys.modules['__main__'].__dict__.pop(_KEY, None)
if prev is not None:
    try:
        prev.close()
    except Exception:
        pass

sys.modules['__main__'].__dict__[_KEY] = vbox
display(vbox)
update()